In [4]:

import os
import subprocess
import shutil
from pathlib import Path
import os

os.environ["LD_LIBRARY_PATH"] = (
    "/mnt/workspace/llama-prebuilt/cuda-12.8:"
 
     "/mnt/workspace/llama-prebuilt/cudart-llama-b11870-bin-ubuntu-cuda-12.8-x64:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

SEARCH_DIR = Path("/mnt/workspace/llama-prebuilt")
MODEL_PATH = Path(
    "/mnt/workspace/saluki/Underdog-Saluki-27B-1.0-IQ2-mix.gguf"
)
LOG = Path("/mnt/workspace/saluki-server-8081.log")

MODEL_NAME = "saluki-27b"
HOST = "127.0.0.1"
PORT = 8081
API_KEY = "saluki-klwjeejklrkjrkejkljrkew"

CTX_SIZE = 200000

servers = [
    p for p in SEARCH_DIR.rglob("llama-server")
    if p.is_file()
    and (
        "cuda-12.8" in str(p).lower()
        or "llama-b11870" in str(p).lower()
    )
]

if not servers:
    raise FileNotFoundError(
        f"未找到 llama-server：{SEARCH_DIR}"
    )

SERVER = servers[0]

if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"找不到模型文件：{MODEL_PATH}"
    )

SERVER.chmod(SERVER.stat().st_mode | 0o111)

if shutil.which("fuser"):
    port_check = subprocess.run(
        ["fuser", "-n", "tcp", str(PORT)],
        capture_output=True,
        text=True
    )
    if port_check.returncode == 0 and port_check.stdout.strip():
        raise RuntimeError(
            f"端口 {PORT} 已被占用：{port_check.stdout.strip()}"
        )

cmd = [
    str(SERVER),
    "-m", str(MODEL_PATH),
    "-ngl", "all",
    "-c", str(CTX_SIZE),
    "-np", "1",
    "--flash-attn", "on",
    "--host", HOST,
    "--port", str(PORT),
    "--api-key", API_KEY,
    "--alias", MODEL_NAME,
]

LOG.parent.mkdir(parents=True, exist_ok=True)
log_file = open(LOG, "w", buffering=1)

process = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
    start_new_session=True
)

print("启动命令已提交")
print("PID：", process.pid)
print("模型：", MODEL_PATH)
print("上下文目标：", CTX_SIZE)
print("API 地址：", f"http://{HOST}:{PORT}/v1")
print("日志：", LOG)


启动命令已提交
PID： 3085
模型： /mnt/workspace/saluki/Underdog-Saluki-27B-1.0-IQ2-mix.gguf
上下文目标： 200000
API 地址： http://127.0.0.1:8081/v1
日志： /mnt/workspace/saluki-server-8081.log


In [2]:
import os
import subprocess
from pathlib import Path

SERVER = Path("/mnt/workspace/llama-prebuilt/cuda-12.8/llama-server")
LIB_DIR = SERVER.parent

# 查找缺失的动态库
matches = list(
    Path("/mnt/workspace/llama-prebuilt").rglob(
        "libllama-server-impl.so"
    )
)

if not matches:
    print("未找到 libllama-server-impl.so")
    print("请检查 llama-server 是否完整解压")
else:
    print("找到动态库：")
    for p in matches:
        print(p)

    # 将动态库所在目录加入搜索路径
    dirs = {str(LIB_DIR)}
    dirs.update(str(p.parent) for p in matches)

    old = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(
        list(dirs) + ([old] if old else [])
    )

    # 检查依赖
    result = subprocess.run(
        ["ldd", str(SERVER)],
        capture_output=True,
        text=True
    )
    print(result.stdout)

    if "not found" not in result.stdout:
        print("动态库检查通过，可以启动 llama-server")
    else:
        print("仍有动态库缺失，请检查 ldd 输出")

找到动态库：
/mnt/workspace/llama-prebuilt/cuda-12.8/libllama-server-impl.so
	linux-vdso.so.1 (0x00007fffa021c000)
	libllama-server-impl.so => /mnt/workspace/llama-prebuilt/cuda-12.8/libllama-server-impl.so (0x00007efe4ea00000)
	libstdc++.so.6 => /lib/x86_64-linux-gnu/libstdc++.so.6 (0x00007efe4e75e000)
	libgcc_s.so.1 => /lib/x86_64-linux-gnu/libgcc_s.so.1 (0x00007efe4f083000)
	libc.so.6 => /lib/x86_64-linux-gnu/libc.so.6 (0x00007efe4e400000)
	libllama-common.so.0 => /mnt/workspace/llama-prebuilt/cuda-12.8/libllama-common.so.0 (0x00007efe4de00000)
	libmtmd.so.0 => /mnt/workspace/llama-prebuilt/cuda-12.8/libmtmd.so.0 (0x00007efe4dc4f000)
	libllama.so.0 => /mnt/workspace/llama-prebuilt/cuda-12.8/libllama.so.0 (0x00007efe4d800000)
	libggml.so.0 => /mnt/workspace/llama-prebuilt/cuda-12.8/libggml.so.0 (0x00007efe4f074000)
	libggml-base.so.0 => /mnt/workspace/llama-prebuilt/cuda-12.8/libggml-base.so.0 (0x00007efe4e695000)
	libssl.so.3 => /lib/x86_64-linux-gnu/libssl.so.3 (0x00007efe4e35c000)
	libc

In [3]:
import os
import subprocess
from pathlib import Path

# 1. 修改为 ROCm 对应的 llama-server 路径
SERVER = Path("/mnt/workspace/llama-prebuilt/rocm/llama-server")
LIB_DIR = SERVER.parent

# 2. 查找 ROCm 环境下缺失的核心动态库或实现库
# ROCm 环境下通常会查找 libllama-server-impl.so 或 hip 相关的动态库
matches = list(
    Path("/mnt/workspace/llama-prebuilt").rglob(
        "libllama-server-impl.so"
    )
)
# 如果还想额外查找其他 ROCm 相关的 so，也可以在这里扩展
hip_matches = list(
    Path("/mnt/workspace/llama-prebuilt").rglob(
        "*.so"
    )
)

if not matches:
    print("未找到核心动态库 libllama-server-impl.so")
    print("请检查 ROCm 版本的 llama-server 是否完整解压")
else:
    print("找到动态库：")
    for p in matches:
        print(p)

    # 3. 构建动态库搜索路径，包含 ROCm 系统路径（/opt/rocm/lib）
    dirs = {str(LIB_DIR)}
    dirs.update(str(p.parent) for p in matches)
    
    # 加上 AMD ROCm 的默认库路径，确保 hipblas、rocblas 等能够被 ldd 找到
    rocm_lib_path = "/opt/rocm/lib"
    if Path(rocm_lib_path).exists():
        dirs.add(rocm_lib_path)

    old = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(
        list(dirs) + ([old] if old else [])
    )

    print(f"\n当前生效的 LD_LIBRARY_PATH: {os.environ['LD_LIBRARY_PATH']}\n")

    # 4. 检查依赖
    result = subprocess.run(
        ["ldd", str(SERVER)],
        capture_output=True,
        text=True
    )
    print("--- ldd 检查结果 ---")
    print(result.stdout)
    
    if "not found" not in result.stdout:
        print("动态库检查通过，ROCm 环境下的 llama-server 可以启动")
    else:
        print("仍有动态库缺失（可能是缺少 /opt/rocm 下的驱动或运行时库），请检查上方 ldd 输出")

找到动态库：
/mnt/workspace/llama-prebuilt/cuda-12.8/libllama-server-impl.so

当前生效的 LD_LIBRARY_PATH: /mnt/workspace/llama-prebuilt/rocm:/opt/rocm/lib:/mnt/workspace/llama-prebuilt/cuda-12.8:/mnt/workspace/llama-prebuilt/cuda-12.8:/mnt/workspace/llama-prebuilt/cuda-12.8:/mnt/workspace/llama-prebuilt/cudart-llama-b11870-bin-ubuntu-cuda-12.8-x64:/opt/rocm/lib:/usr/local/lib:

--- ldd 检查结果 ---

动态库检查通过，ROCm 环境下的 llama-server 可以启动
